# Lab 2 — 스팸 필터의 원형 (Naive Bayes)

**확률통계 · Topic 2 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 데이터를 **세어서** 조건부 확률 $\mathbb{P}[\text{word} \mid \text{spam}]$ 을 구한다.
2. **Bayes 정리**로 방향을 뒤집어 $\mathbb{P}[\text{spam} \mid \text{words}]$ 를 계산한다.
3. 확률이 **0이 되어버리는 문제**를 직접 만난다. (해결은 Topic 13)

⏱ **예상 소요 시간: 35분**

## Part 0. 데이터

진짜 메일 데이터셋은 크다. 오늘은 **손으로 셀 수 있는 크기**로 시작한다.
각 메일은 소문자 단어들로 이루어져 있고, 라벨은 `spam` 또는 `ham`(정상)이다.

In [ ]:
import numpy as np

rng = np.random.default_rng(20260302)

mails = [
    ("spam", "free money click now win"),
    ("spam", "win free prize click link"),
    ("spam", "urgent free offer click here"),
    ("spam", "you win money now urgent"),
    ("spam", "click link free lottery win"),
    ("spam", "cheap offer free shipping the deal"),
    ("ham",  "meeting tomorrow at ten"),
    ("ham",  "please review the report"),
    ("ham",  "lunch at noon today"),
    ("ham",  "the lecture note is uploaded"),
    ("ham",  "can you send me the free file"),
    ("ham",  "homework deadline is friday"),
    ("ham",  "thanks for the quick review"),
    ("ham",  "see you at the meeting"),
    ("ham",  "please click the attached file"),
]

spam_mails = [t.split() for label, t in mails if label == "spam"]
ham_mails = [t.split() for label, t in mails if label == "ham"]

print("스팸", len(spam_mails), "통 / 정상", len(ham_mails), "통")

## Part 1. 세어서 확률 구하기

먼저 **prior** 부터. 아무 정보 없이 메일 하나를 집었을 때 스팸일 확률이다.

### 실습 1 — prior 계산

In [ ]:
n_spam, n_ham = len(spam_mails), len(ham_mails)
n_total = n_spam + n_ham

p_spam = 0.5   # TODO 1: 전체 중 스팸의 비율로 바꾸세요
p_ham = 0.5    # TODO 1: 전체 중 정상 메일의 비율로 바꾸세요

print(f"P[spam] = {p_spam:.3f},  P[ham] = {p_ham:.3f}")

### 실습 2 — likelihood 계산

$\mathbb{P}[\text{word} \mid \text{spam}]$ = **그 단어가 등장한 스팸 메일 수 / 전체 스팸 메일 수**

> 단어가 몇 번 나왔는지가 아니라 **몇 통에 나왔는지**로 센다.

In [ ]:
def word_prob(word, mails_list):
    # TODO 2: 그 단어를 포함한 메일 수를 세세요
    #         힌트: sum(1 for words in mails_list if word in words)
    count = 0
    return count / len(mails_list)


for w in ["free", "click", "the"]:
    print(f"{w:>8} | P[w|spam] = {word_prob(w, spam_mails):.3f}"
          f" | P[w|ham] = {word_prob(w, ham_mails):.3f}")

`free`, `click` 은 스팸 쪽이 훨씬 높고, `the` 는 양쪽에 골고루 나온다.
**판별력이 있는 단어와 없는 단어**가 나뉜다.

## Part 2. Bayes로 방향 뒤집기

우리가 원하는 것은 $\mathbb{P}[\text{spam} \mid \text{word}]$ 다.

$$\mathbb{P}[\text{spam} \mid w] = \frac{\mathbb{P}[w \mid \text{spam}]\,\mathbb{P}[\text{spam}]}
{\mathbb{P}[w \mid \text{spam}]\mathbb{P}[\text{spam}] + \mathbb{P}[w \mid \text{ham}]\mathbb{P}[\text{ham}]}$$

분모가 바로 **전확률 법칙**이다.

### 실습 3 — 단어 하나로 판정하기

In [ ]:
def posterior_one_word(word):
    lik_spam = word_prob(word, spam_mails) * p_spam
    lik_ham = word_prob(word, ham_mails) * p_ham
    if lik_spam + lik_ham == 0:
        return None
    # TODO 3: Bayes 정리로 P[spam | word] 를 계산해 돌려주세요
    return 0.0


for w in ["free", "click", "the"]:
    print(f"{w:>8} -> P[spam|w] = {posterior_one_word(w)}")

## Part 3. 단어 여러 개 — 조건부 독립 가정

메일 전체를 쓰려면 단어를 여러 개 봐야 한다. 그런데
$\mathbb{P}[w_1, w_2, \ldots \mid \text{spam}]$ 을 직접 세려면
**그 단어 조합이 통째로 등장한 메일**이 필요하다. 데이터가 아무리 많아도 부족하다.

그래서 **조건부 독립을 가정**한다 — 이것이 Naive Bayes의 "naive".

$$\mathbb{P}[w_1, \ldots, w_k \mid \text{spam}] \approx \prod_{i=1}^{k} \mathbb{P}[w_i \mid \text{spam}]$$

### 실습 4 — 메일 전체로 판정하기

In [ ]:
def classify(text):
    words = text.split()
    score_spam, score_ham = p_spam, p_ham
    for w in words:
        # TODO 4: 각 단어의 likelihood를 곱해 나가세요
        #         score_spam 에는 P[w|spam] 을, score_ham 에는 P[w|ham] 을 곱합니다
        pass
    if score_spam + score_ham == 0:
        return None
    return score_spam / (score_spam + score_ham)


for t in ["free click", "the meeting", "free money pizza"]:
    print(f"{t:>20} -> P[spam] = {classify(t)}")

## Part 4. 결과 읽기 — 왜 0 아니면 1에 가까운가

실습 4의 출력을 보자. 세 문장이 **각각 다른 이유로** 다른 답을 냈다.

| 입력 | 결과 | 이유 |
|---|---|---|
| `free click` | 0.9 이상 | 두 단어 다 스팸 쪽 likelihood가 높다 — 곱하니 더 벌어졌다 |
| `the meeting` | 0.0 | `meeting` 이 스팸 메일에 **한 번도** 안 나왔다 |
| `free money pizza` | None | `pizza` 를 양쪽 모두 본 적이 없다 → 점수가 둘 다 0 |

단어를 곱할수록 확신이 **극단으로 치우친다.** 이것은 Naive Bayes의 알려진 성질이다
(조건부 독립 가정이 틀렸는데 증거를 독립인 것처럼 여러 번 세기 때문).

그리고 더 심각한 문제가 있다. 학습 데이터에 **없던 단어** 하나가 전체를 무너뜨린다.

In [ ]:
print("free money meeting  ->", classify("free money meeting"))
print("free money pizza    ->", classify("free money pizza"))   # 'pizza'는 처음 보는 단어

둘 다 `None` 이 나왔다. **판정 자체가 불가능**해진 것이다.

- `free money pizza` — `pizza` 를 양쪽 모두 본 적이 없어 두 점수가 함께 0이 된다.
- `free money meeting` — `meeting` 때문에 스팸 점수가 0, `money` 때문에 정상 점수가 0이 된다.
  스팸스러운 단어가 둘이나 있는데도 **판정을 포기**하게 된다.

> ❗ **단어 하나가 거부권처럼 전체를 무너뜨린다.**
> 해결책은 "한 번도 못 봤다고 확률이 0인 것은 아니다"라고 보정하는 것 —
> **Laplace smoothing**, 그리고 그것이 사실 **Beta prior**라는 사실은 **Topic 13**에서 다룬다.
> 오늘은 문제를 확인하는 데까지가 목표다.

## Part 5. 몬티 홀 후속 — 정보의 출처가 다르면

지난주에는 진행자가 **답을 알고** 염소 문을 열었다.
만약 진행자도 모른 채 **무작위로** 열었는데 우연히 염소였다면 승률이 달라질까?

### 실습 5

In [ ]:
def monty_random_host(n_trials=20000, seed=20260302):
    rng = np.random.default_rng(seed)
    switch_wins = stay_wins = valid = 0

    for _ in range(n_trials):
        prize = rng.integers(0, 3)
        choice = rng.integers(0, 3)
        # 진행자가 내 문을 뺀 나머지 중 아무거나 연다 (상품일 수도 있다!)
        others = [d for d in range(3) if d != choice]
        opened = rng.choice(others)

        # TODO 5: 진행자가 상품 문을 열어버린 판은 건너뛰세요
        #         힌트: if opened == prize: continue
        if opened == prize:
            pass
        valid += 1

        remaining = [d for d in range(3) if d != choice and d != opened][0]
        if remaining == prize:
            switch_wins += 1
        if choice == prize:
            stay_wins += 1

    return stay_wins / valid, switch_wins / valid, valid


stay, switch, valid = monty_random_host()
print(f"유효한 판: {valid}")
print(f"유지 승률 {stay:.3f} / 교체 승률 {switch:.3f}")

🤔 **결과가 지난주와 다른가?**

진행자가 답을 알고 열면 교체가 **2/3**, 모르고 열었는데 우연히 염소였다면 **1/2**이다.
눈에 보이는 장면(염소 문이 열렸다)은 똑같은데 확률이 다르다.

> **확률은 눈에 보이는 결과뿐 아니라, 그 결과가 만들어진 과정에 달려 있다.**
> 데이터가 어떻게 수집되었는지 모르면 확률을 계산할 수 없다 — Topic 14 함정의 예고편이다.

---

## 마무리 — 자가 점검

- [ ] 데이터를 세어서 조건부 확률을 구할 수 있다
- [ ] Bayes 정리로 조건의 방향을 뒤집을 수 있다
- [ ] Naive Bayes의 "naive"가 무슨 가정인지 설명할 수 있다
- [ ] 확률이 0이 되는 문제를 직접 확인했다
- [ ] 정보의 생성 과정이 확률을 바꾼다는 것을 몬티 홀로 확인했다

**오늘 가장 놀라웠던 점을 한 문장으로.**

> (여기에 작성)